In [1]:
# ============================================================
# QCTrojan-Bench: Quantum Circuit Trojan Detection Benchmark
# ============================================================
# Notebook:  S4_split_index.ipynb
# Purpose:   Generate source-circuit stratified split index
#            for all downstream model training notebooks.
#
#            All model notebooks (S5, S6) must load this
#            split index and never generate their own splits.
#
# Authors:   Zeeshan Ajmal
#            University of Oulu, Finland
# Version:   QCTrojan-Bench v1.0
# License:   CC BY 4.0
# ============================================================
#
# CELL 1 — Imports and Configuration
#
# Prerequisites:
#   S3_feature_extraction.ipynb complete
#   dataset/features/features_v1.csv must exist
# ============================================================

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from sklearn.model_selection import StratifiedKFold

# ── Versioning ───────────────────────────────────────────────
DATASET_VERSION = "v1"

# ── Paths (auto-detected) ─────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATASET_ROOT = PROJECT_ROOT / "dataset"
FEATURES_DIR = DATASET_ROOT / "features"

FEATURES_PATH = FEATURES_DIR / "features_v1.csv"
SPLIT_PATH    = FEATURES_DIR / "split_index.csv"
MANIFEST_PATH = FEATURES_DIR / "split_manifest.json"

# ── Split configuration ───────────────────────────────────────
SEED      = 42
N_FOLDS   = 5
N_REPEATS = 3

# ── Load features ─────────────────────────────────────────────
assert FEATURES_PATH.exists(), \
    f"features_v1.csv not found at {FEATURES_PATH}"

df = pd.read_csv(FEATURES_PATH)

print("=" * 55)
print("QCTrojan-Bench — S4 Split Index Generation")
print("=" * 55)
print(f"  Seed           : {SEED}")
print(f"  Folds          : {N_FOLDS}")
print(f"  Repeats        : {N_REPEATS}")
print(f"  Total CV fits  : {N_FOLDS * N_REPEATS} per model")
print(f"  Features path  : {FEATURES_PATH}")
print(f"  Dataset shape  : {df.shape}")
print(f"  Label counts   : "
      f"{df['trojan_type'].value_counts().to_dict()}")
print("=" * 55)
print("Cell 1 complete. Ready for Cell 2.")

QCTrojan-Bench — S4 Split Index Generation
  Seed           : 42
  Folds          : 5
  Repeats        : 3
  Total CV fits  : 15 per model
  Features path  : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\features\features_v1.csv
  Dataset shape  : (9000, 45)
  Label counts   : {'benign': 3000, 'static': 3000, 'triggered': 3000}
Cell 1 complete. Ready for Cell 2.


In [2]:
# ============================================================
# CELL 2 — Build Source-Stratified Split Index
# ============================================================
#
# Source-circuit stratification:
#   Each benign circuit has exactly 2 tampered variants.
#   All 3 variants share the same source_id.
#   The split is performed at source level so all 3 variants
#   of a source circuit always stay on the same side of the
#   train/test boundary.
#
#   This prevents lineage leakage — a tampered circuit
#   cannot appear in the test set if its benign parent
#   is in the training set.
#
# IID CV splits:
#   5-fold × 3 repeats = 15 evaluation points per model.
#   Stratified by algorithm_family at the source level.
#
# Holdout splits:
#   5 rounds of leave-one-family-out.
#   Each round holds out one algorithm family completely.
# ============================================================

# ── Build source_id ───────────────────────────────────────────
# Benign rows:   source_id = own sample_id
# Tampered rows: source_id = parent_sample_id

df["source_id"] = df.apply(
    lambda r: r["sample_id"]
    if r["label"] == "benign"
    else r["parent_sample_id"],
    axis=1,
)

# Verify each source_id maps to exactly 3 circuits
counts = df.groupby("source_id").size()
assert (counts == 3).all(), \
    f"Expected 3 circuits per source. Got:\n{counts[counts != 3]}"

n_sources = df["source_id"].nunique()
print(f"Source circuits      : {n_sources} (expected 3000)")
print(f"Circuits per source  : 3 (benign + static + triggered) ✓")

# ── Build source-level dataframe ──────────────────────────────
# One row per source circuit, stratified by family

source_df = (
    df[df["label"] == "benign"][["source_id", "algorithm_family"]]
    .reset_index(drop=True)
    .copy()
)

print(f"\nSource-level distribution:")
print(source_df["algorithm_family"].value_counts().to_string())

# ── Assign IID CV fold labels ─────────────────────────────────
# 5-fold × 3 repeats

rng = np.random.RandomState(SEED)

for repeat in range(N_REPEATS):
    # Shuffle source order differently each repeat
    shuffled_idx = rng.permutation(len(source_df))
    source_shuffled = source_df.iloc[shuffled_idx].reset_index(
        drop=True
    )

    skf = StratifiedKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=SEED + repeat,
    )

    fold_col = np.zeros(len(source_df), dtype=int)

    for fold_idx, (_, test_idx) in enumerate(
        skf.split(
            source_shuffled,
            source_shuffled["algorithm_family"],
        )
    ):
        original_positions = shuffled_idx[test_idx]
        fold_col[original_positions] = fold_idx

    source_df[f"fold_iid_r{repeat}"] = fold_col

fold_cols = [f"fold_iid_r{r}" for r in range(N_REPEATS)]

print(f"\nIID fold assignments created ({N_REPEATS} repeats):")
for r in range(N_REPEATS):
    dist = source_df[f"fold_iid_r{r}"].value_counts().sort_index()
    print(f"  Repeat {r}: {dist.to_dict()}")

# ── Assign holdout family labels ──────────────────────────────
# For leave-one-family-out evaluation
# holdout_family = the family this source belongs to
# Used by S5: train on rows where family != holdout_family

source_df["holdout_family"] = source_df["algorithm_family"]

# ── Merge back to circuit level ───────────────────────────────

split_cols = ["source_id"] + fold_cols + ["holdout_family"]

split_index = df[
    ["sample_id", "source_id", "algorithm_family",
     "label", "trojan_type"]
].copy()

split_index = split_index.merge(
    source_df[split_cols],
    on="source_id",
    how="left",
)

print(f"\nSplit index shape : {split_index.shape}")
print(f"NaN check         : "
      f"{split_index[fold_cols].isnull().sum().sum()} "
      f"(should be 0)")

# ── Verify zero leakage ───────────────────────────────────────
print("\nVerifying zero source-circuit leakage...")

leakage_errors = 0

for r in range(N_REPEATS):
    col = f"fold_iid_r{r}"
    for fold in range(N_FOLDS):
        test_sources  = set(
            split_index.loc[split_index[col] == fold, "source_id"]
        )
        train_sources = set(
            split_index.loc[split_index[col] != fold, "source_id"]
        )
        overlap = test_sources & train_sources
        if overlap:
            print(f"  LEAKAGE: repeat={r} fold={fold} "
                  f"overlap={len(overlap)} sources")
            leakage_errors += 1

if leakage_errors == 0:
    print(f"  Zero leakage across all "
          f"{N_REPEATS * N_FOLDS} fold-repeat combinations ✓")
else:
    raise RuntimeError(
        f"{leakage_errors} leakage violations detected."
    )

# ── Verify family stratification ──────────────────────────────
print("\nVerifying family stratification per fold...")

strat_ok = True
for r in range(N_REPEATS):
    col = f"fold_iid_r{r}"
    for fold in range(N_FOLDS):
        test_mask    = split_index[col] == fold
        test_families = (
            split_index.loc[
                test_mask & (split_index["label"] == "benign"),
                "algorithm_family"
            ].value_counts()
        )
        if len(test_families) < 5:
            print(f"  WARNING: repeat={r} fold={fold} "
                  f"missing families: {test_families.to_dict()}")
            strat_ok = False

if strat_ok:
    print("  All 5 families present in every fold ✓")

# ── Save split index ──────────────────────────────────────────
split_index.to_csv(SPLIT_PATH, index=False)
print(f"\nSaved: {SPLIT_PATH}")
print("Cell 2 complete. Ready for Cell 3.")

Source circuits      : 3000 (expected 3000)
Circuits per source  : 3 (benign + static + triggered) ✓

Source-level distribution:
algorithm_family
deutsch_jozsa    600
grover           600
qaoa             600
vqc              600
qft              600

IID fold assignments created (3 repeats):
  Repeat 0: {0: 600, 1: 600, 2: 600, 3: 600, 4: 600}
  Repeat 1: {0: 600, 1: 600, 2: 600, 3: 600, 4: 600}
  Repeat 2: {0: 600, 1: 600, 2: 600, 3: 600, 4: 600}

Split index shape : (9000, 9)
NaN check         : 0 (should be 0)

Verifying zero source-circuit leakage...
  Zero leakage across all 15 fold-repeat combinations ✓

Verifying family stratification per fold...
  All 5 families present in every fold ✓

Saved: C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\features\split_index.csv
Cell 2 complete. Ready for Cell 3.


In [4]:
# ============================================================
# CELL 3 — Save Manifest and Print Summary
# ============================================================

manifest = {
    "notebook":          "S4_split_index.ipynb",
    "created":           datetime.now(timezone.utc).isoformat(),
    "seed":              SEED,
    "n_folds":           N_FOLDS,
    "n_repeats":         N_REPEATS,
    "total_cv_fits":     N_FOLDS * N_REPEATS,
    "total_circuits":    len(df),
    "total_sources":     n_sources,
    "features_path":     str(FEATURES_PATH),
    "split_path":        str(SPLIT_PATH),
    "stratification":    "source_circuit_by_algorithm_family",
    "leakage_verified":  True,
    "family_counts": (
        df["algorithm_family"].value_counts().to_dict()
    ),
    "trojan_type_counts": (
        df["trojan_type"].value_counts().to_dict()
    ),
    "fold_cols":         fold_cols,
    "holdout_col":       "holdout_family",
    "validation_passed": True,
}

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

print("=" * 55)
print("S4 — Summary")
print("=" * 55)
print(f"  Total circuits   : {len(df)}")
print(f"  Total sources    : {n_sources}")
print(f"  CV folds         : {N_FOLDS} × {N_REPEATS} repeats "
      f"= {N_FOLDS * N_REPEATS} fits per model")
print(f"  Holdout rounds   : 5 (one per family)")
print(f"  Leakage verified : zero ✓")
print(f"  Split saved      : {SPLIT_PATH}")
print(f"  Manifest saved   : {MANIFEST_PATH}")
print("=" * 55)
print("\nS4 complete. Ready for S5 — model training.")


S4 — Summary
  Total circuits   : 9000
  Total sources    : 3000
  CV folds         : 5 × 3 repeats = 15 fits per model
  Holdout rounds   : 5 (one per family)
  Leakage verified : zero ✓
  Split saved      : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\features\split_index.csv
  Manifest saved   : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\features\split_manifest.json

S4 complete. Ready for S5 — model training.
